# AdaptiveDepth Phase 7.2 semantic label review
Review only the 128 ambiguous Route 20/23 examples with a conservative 3B judge. Use a T4 GPU. The output is resumable and saved directly to Google Drive.

In [ ]:
import base64, os, pathlib, subprocess, sys
REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
repo = pathlib.Path('/content/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs')
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if not repo.exists():
    env = os.environ.copy()
    token = secret('GITHUB_TOKEN')
    if token:
        credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env.update({'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.extraHeader','GIT_CONFIG_VALUE_0':f'Authorization: Basic {credential}'})
    result = subprocess.run(['git','clone',REPO_URL,str(repo)],env=env,capture_output=True,text=True)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or 'Git clone failed; add GITHUB_TOKEN to Colab Secrets')
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-e','.','accelerate>=0.26'],check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = pathlib.Path('/content/drive/MyDrive/AdaptiveDepth-results')
drive_root.mkdir(parents=True,exist_ok=True)
review = drive_root / 'phase72_label_review.jsonl'
judgments = drive_root / 'phase72_label_judgments.jsonl'
if not review.is_file():
    raise FileNotFoundError(f'Upload phase72_label_review.jsonl to {drive_root}')
print('Review rows:',sum(1 for line in review.open(encoding='utf-8') if line.strip()))
print('Resuming judgments at:',judgments)

In [ ]:
command = [
    'adaptivedepth','judge-phase72-label-review',
    '--review',str(review),
    '--output',str(judgments),
    '--model-id','Qwen/Qwen2.5-3B-Instruct',
    '--resume',
]
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line,end='')
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Judge command failed with exit code {return_code}; see the detailed output above')

In [ ]:
import json, collections
rows=[json.loads(line) for line in judgments.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Completed:',len(rows),'/ 128')
print('Verdicts:',collections.Counter(row['verdict'] for row in rows))
print('High-confidence safe:',sum(row['verdict']=='safe_route' and row['confidence']>=0.8 for row in rows))
print('Saved:',judgments)